# Kiva Lending Patterns - Data Cleaning, EDA and Dashboard

Jupyter notebook presenting lending pattern analysis for Kiva loans. The analysis includes data cleaning, exploratory data analysis (EDA), and an interactive dashboard built with Dash. A brief setup section at the beginning ensures that the latest versions of all necessary libraries are installed and imported, and the paths for the data files are correctly set. Adjust this section and the file paths as needed to match your local environment.



---
## Section 0: Kill process listening on Dash default port (8050) to avoid conflicts

This step kills any process listening to the default Dash port (8050) to avoid conflicts when running the dashboard (which happens if another process was started earlier). This may kill the python kernel needed to run the notebook and a dialog may appear asking to restart the kernel. 

In [ ]:
import psutil

def kill_port(port):
    """Kill any process listening on the given port."""
    killed = False
    for conn in psutil.net_connections():
        if conn.laddr.port == port and conn.status == psutil.CONN_LISTEN:
            try:
                proc = psutil.Process(conn.pid)
                proc.terminate()
                proc.wait(timeout=3)
                print(f'Process {conn.pid} ({proc.name()}) on port {port} has been terminated.')
                killed = True
            except psutil.NoSuchProcess:
                pass  # Process already gone; port is free
            except (psutil.AccessDenied, psutil.TimeoutExpired) as e:
                print(f'Could not terminate process {conn.pid}: {e}')
    if not killed:
        print(f'No process is listening on port {port}.')


Execute the following cell only if you want to kill the process listening on the default Dash port (8050).

In [ ]:


# kill_port(8050)


---

## Section 1: Setup

All required libraries are installed or upgraded. Then the necessary libraries are imported.

In [ ]:

%pip install --upgrade pandas numpy seaborn plotly dash


In [ ]:


import os
import pandas as pd
import numpy as np
import plotly.express as px


File paths for the raw input data and the cleaned output data are defined below. These paths are used in the subsequent data processing and analysis steps.

In [ ]:


# Update DATA_INPUT_PATH if running from a different directory
DATA_INPUT_PATH   = os.path.join('..', 'data', 'data_abschlussprojekt.csv')
DATA_CLEANED_PATH = os.path.join('..', 'data', 'kiva_cleaned.csv')
print('Paths set.')


---

## Section 2: Data Loading and Cleaning

The raw dataset is loaded and inspected. Cleaning steps are applied in sequence, and the cleaned CSV is saved.

# Business Case: Kiva Lending Patterns
## An Exploratory Data Analysis and Interactive Dashboard

### Project Objective

Kiva's historical lending data is explored to identify patterns in loan
distribution across sectors and countries, and to investigate differences
between requested and funded loan amounts.

Data inspection and cleaning, exploratory data visualization, a funding
gap comparison, and an interactive dashboard developed using Dash are
included in this analysis.

### 1. Loading the Dataset

The file uses `#` as the column separator. The first column is an unnamed row index; `index_col=0` is used so that it becomes the DataFrame index. `skipinitialspace=True` is applied so that leading spaces after each `#` are stripped from the header, yielding clean column names without further post-processing.

In [ ]:
# Load the dataset

df = pd.read_csv(
    DATA_INPUT_PATH,
    sep='#',
    index_col=0,
    skipinitialspace=True
)

# print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head(5)

### 2. Initial Inspection

Before any cleaning, the raw state of the data is inspected: column names, data types, first rows, summary statistics, missing values, and duplicates.

In [ ]:
# Display the shape of the DataFrame
df.shape

In [ ]:
# Display the column names of the DataFrame
df.columns.to_list()  # Convert the column names to a list for easier inspection

In [ ]:
# Display the data types of each column in the DataFrame
df.dtypes

In [ ]:
# Display concise summary of the DataFrame, including column data types and non-null counts
df.info()

In [ ]:
# Display summary statistics for all columns, including categorical and numerical data
df.describe(include='all')

#### Missing Values

The count and percentage of missing values are reported per column.

In [ ]:
# Calculate and display missing values for each column in the DataFrame
# Calculate missing counts and percentages
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df) * 100).round(2)

# Combine into a new DataFrame
missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_pct
})

# Filter to show only columns that have at least one missing value
missing_df = missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing %', ascending=False)

# Display the result
print("Columns with missing values:")
# print(missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing %', ascending=False))
display(missing_df)


#### Duplicate Rows

In [ ]:
# Display the count of unique values for each column
df.nunique()

In [ ]:
# Display all duplicated rows in the DataFrame
df[df.duplicated(keep=False)]

In [ ]:
# Check for exact duplicate rows in the DataFrame
n_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {n_dupes:,}")  
print(f"Unique rows: {len(df) - n_dupes:,} of {len(df):,} ({(1 - n_dupes/len(df))*100:.2f}%)")  # Print the number and percentage of unique rows

#### Categorical Column Values

Unique values in key categorical columns are checked to identify encoding inconsistencies before cleaning.

In [ ]:
print("Unique sectors:", sorted(df['sector'].dropna().unique()))
print("\nUnique repayment intervals:", sorted(df['repayment_interval'].dropna().unique()))

#### Numerical Column Ranges

Columns that should be strictly positive are checked for negative or zero values.

In [ ]:
for col in ['funded_amount', 'loan_amount', 'term_in_months', 'lender_count']:
    neg = (df[col] < 0).sum()
    zero = (df[col] == 0).sum()
    print(f"{col}: min={df[col].min()}, max={df[col].max()}, zeros={zero:,}, negatives={neg:,}")

In [ ]:
# Create a data dictionary containing column names, data types, missing values, and unique values
data_dictionary = pd.DataFrame({
    "Column": df.columns,
    "dtype": df.dtypes.values,
    "Data type": [
        type(df[col].iloc[0]).__name__
        for col in df.columns
    ],
    "Missing Values": df.isnull().sum().values,
    "Unique Values": df.nunique().values
    
})
data_dictionary

### 3. Data Cleaning

The following cleaning steps are applied in sequence. The decision and justification for each step are documented in the cell below it.

#### Step 1 - Strip Whitespace from String Columns

String values in the raw file may carry leading or trailing whitespace. This is stripped uniformly across all object columns without losing any data.

In [ ]:
str_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
for col in str_cols:
    df[col] = df[col].str.strip()
print(f"Stripped whitespace from {len(str_cols)} string columns: {str_cols}")

#### Step 2 - Duplicate Rows

Exact duplicate rows are dropped only if confirmed. A row is an exact duplicate if every column value matches another row. Kiva loan records are not expected to repeat; genuine duplicates are data entry errors.

In [ ]:
n_before = len(df)
df = df.drop_duplicates()
n_dropped = n_before - len(df)
print(f"Rows before: {n_before:,}")
print(f"Exact duplicates dropped: {n_dropped:,}")
print(f"Rows after: {len(df):,}")

#### Step 3 - Missing Values: `region`

`region` is a free-text field describing the sub-national area within a country. Kiva data commonly has a large share of missing regions because lenders are not required to supply them. This step records the count; the fill to `'Not specified'` happens in Step 9 together with all remaining string columns.

In [ ]:
missing_region = df['region'].isnull().sum()
pct_region = missing_region / len(df) * 100
print(f"Missing region: {missing_region:,} ({pct_region:.1f}% of rows)")
print("Will be filled with 'Not specified' in Step 9.")

#### Step 4 - Missing Values: `borrower_genders`

`borrower_genders` holds one or more gender labels separated by commas. Missing values indicate the information was not supplied. This step records the count; the fill happens in Step 9.

In [ ]:
missing_bg = df['borrower_genders'].isnull().sum()
pct_bg = missing_bg / len(df) * 100
print(f"Missing borrower_genders: {missing_bg:,} ({pct_bg:.1f}% of rows)")
print("Will be filled with 'Not specified' in Step 9.")

#### Step 5 - Missing Values: `use`

`use` is a free-text description of how the borrower intends to use the loan. This step records the count; the fill happens in Step 9.

In [ ]:
missing_use = df['use'].isnull().sum()
pct_use = missing_use / len(df) * 100
print(f"Missing use: {missing_use:,} ({pct_use:.1f}% of rows)")
print("Will be filled with 'Not specified' in Step 9.")

#### Step 6 - Missing Values: `country_code`

Eight rows have a missing `country_code` but a populated `country` value of 'Namibia'. The ISO 3166-1 alpha-2 code for Namibia is 'NA'. These eight rows are filled rather than dropped because the country is unambiguously known.

In [ ]:
missing_cc = df['country_code'].isnull().sum()
print(f"Missing country_code before fill: {missing_cc:,}")
print("Countries for these rows:")
print(df.loc[df['country_code'].isnull(), 'country'].value_counts().to_string())

df.loc[df['country_code'].isnull() & (df['country'] == 'Namibia'), 'country_code'] = 'NA'

remaining_cc = df['country_code'].isnull().sum()
print(f"\nMissing country_code after fill: {remaining_cc:,}")

#### Step 7 - Missing Values: `term_in_months`

`term_in_months` is checked for missing values. If present, they are retained rather than imputed, since imputing duration without supporting data would distort the distribution.

In [ ]:
missing_term = df['term_in_months'].isnull().sum()
pct_term = missing_term / len(df) * 100
print(f"Missing term_in_months: {missing_term:,} ({pct_term:.1f}% of rows)")

#### Step 8 - Remaining Missing Values Check

Columns with remaining NaN values are identified before the fill step.

In [ ]:
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
if remaining_missing.empty:
    print("No NaN remaining before fill step.")
else:
    print("Columns with NaN before fill:")
    for col, cnt in remaining_missing.items():
        print(f"  {col}: {cnt:,} ({cnt/len(df)*100:.2f}%)")

#### Step 9 - Fill Remaining String NaN with 'Not specified'

All remaining NaN values in string columns are filled with `'Not specified'`. This makes the absence of data explicit and visible in groupby operations and visualizations instead of being silently excluded. The label `'Not specified'` is chosen over `'Unknown'` to avoid implying the information exists but is unclear -- it simply was not recorded.

In [ ]:
str_cols_with_nan = df.select_dtypes(include=['object', 'str']).columns[
    df.select_dtypes(include=['object', 'str']).isnull().any()
].tolist()

for col in str_cols_with_nan:
    filled = df[col].isnull().sum()
    df[col] = df[col].fillna('Not specified')
    print(f"  {col}: filled {filled:,} NaN with 'Not specified'")

print(f"\nTotal NaN remaining in string columns: {df.select_dtypes(include=['object', 'str']).isnull().sum().sum()}")

#### Step 10 - Numerical Validity

Records with a `loan_amount` of zero or below are not meaningful loan records. `funded_amount` can legitimately be zero (partially funded) but not negative. Rows failing these checks are dropped with a count reported.

In [ ]:
n_before = len(df)

invalid_loan = df['loan_amount'] <= 0
invalid_funded = df['funded_amount'] < 0
invalid_mask = invalid_loan | invalid_funded

print(f"Rows with loan_amount <= 0:  {invalid_loan.sum():,}")
print(f"Rows with funded_amount < 0: {invalid_funded.sum():,}")
print(f"Total invalid rows to drop:  {invalid_mask.sum():,}")

df = df[~invalid_mask].copy()
print(f"\nRows after numerical validity filter: {len(df):,} (dropped {n_before - len(df):,})")

#### Step 11 - Unusual `term_in_months` Values

The distribution of `term_in_months` is inspected to identify unusually short or long durations. Unusual values are noted but not removed unless they are clearly data entry errors (negative or zero), since genuine long-term loans exist.

In [ ]:
print("term_in_months distribution:")
print(df['term_in_months'].describe())
print(f"\nRows with term_in_months <= 0: {(df['term_in_months'] <= 0).sum():,}")
print(f"Rows with term_in_months > 120 (more than 10 years): {(df['term_in_months'] > 120).sum():,}")

Rows where `term_in_months` is zero or negative are dropped, as these cannot represent valid loan durations.

In [ ]:
n_before = len(df)
invalid_term = df['term_in_months'].notna() & (df['term_in_months'] <= 0)
df = df[~invalid_term].copy()
print(f"Dropped {n_before - len(df):,} rows with term_in_months <= 0.")
print(f"Rows remaining: {len(df):,}")

#### Step 12 - Derived Feature: `funding_gap`

`funding_gap` is the difference between the requested amount and the funded amount. A positive value means the loan was partially funded. A negative value means more was disbursed than originally requested; the assignment context confirms this can happen and these records are kept. Zero means fully funded.

In [ ]:
df['funding_gap'] = df['loan_amount'] - df['funded_amount']

fully_funded = (df['funding_gap'] == 0).sum()
partially_funded = (df['funding_gap'] > 0).sum()
over_funded = (df['funding_gap'] < 0).sum()

print(f"Fully funded (gap = 0):          {fully_funded:,} ({fully_funded/len(df)*100:.1f}%)")
print(f"Partially funded (gap > 0):      {partially_funded:,} ({partially_funded/len(df)*100:.1f}%)")
print(f"Over-funded (gap < 0):           {over_funded:,} ({over_funded/len(df)*100:.1f}%)")

print("\nfunding_gap statistics:")
print(df['funding_gap'].describe())

#### Step 13 - Final Cleaned Dataset Summary

In [ ]:
print("Final cleaned dataset:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {df.columns.tolist()}")
print("\nRemaining missing values:")
final_missing = df.isnull().sum()
print(final_missing[final_missing > 0])
print("\nFirst rows:")
df.head()

In [ ]:
# Display the shape of the DataFrame after cleaning
df.shape

In [ ]:
# Create a data dictionary containing column names, data types, missing values, and unique values after cleaning
data_dictionary = pd.DataFrame({
    "Column": df.columns,
    "dtype": df.dtypes.values,
    "Data type": [
        type(df[col].iloc[0]).__name__
        for col in df.columns
    ],
    "Missing Values": df.isnull().sum().values,
    "Unique Values": df.nunique().values
    
})
data_dictionary

### 4. Save Cleaned Dataset

The cleaned and processed dataset is saved as a CSV to the `data` directory for use in subsequent analysis and visualization steps.

In [ ]:
df.to_csv(DATA_CLEANED_PATH)
print(f"Saved cleaned dataset to: {DATA_CLEANED_PATH}")
print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")

In [ ]:
# Make dfc available for the visualization and dashboard sections
dfc = df.copy()
print(f'dfc ready: {dfc.shape[0]:,} rows, {dfc.shape[1]} columns')


---

## Section 3: Assessed Visualizations

Three charts selected as the assessed deliverables.

In [ ]:
TOP_N_COUNTRIES = 10
country_counts = dfc['country'].value_counts().head(TOP_N_COUNTRIES).reset_index()
country_counts.columns = ['country', 'count']
country_counts['pct'] = (country_counts['count'] / len(dfc) * 100).round(1)

fig_a1 = px.bar(
    country_counts,
    x='count',
    y='country',
    orientation='h',
    text=country_counts['pct'].map('{:.1f}%'.format),
    custom_data=['pct'],
    title=f'Top {TOP_N_COUNTRIES} Countries by Number of Kiva Loans',
    labels={'country': 'Country', 'count': 'Number of Loans'}
)
fig_a1.update_traces(
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Number of loans: %{x:,}<br>'
        '% of total number of loans: %{customdata[0]:.1f}%'
        '<extra></extra>'
    )
)
fig_a1.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_a1.show()

In [ ]:
# Number of top countries to display -- increase or decrease as needed
TOP_N_COUNTRIES = 10

top_countries = dfc['country'].value_counts().head(TOP_N_COUNTRIES).index.tolist()
top_df = dfc[dfc['country'].isin(top_countries)].copy()

sector_by_country = (
    top_df.groupby(['country', 'sector'])
    .size()
    .reset_index(name='count')
)
country_totals = top_df.groupby('country').size().reset_index(name='total')
sector_by_country = sector_by_country.merge(country_totals, on='country')
sector_by_country['pct'] = sector_by_country['count'] / sector_by_country['total'] * 100
sector_by_country['country_label'] = sector_by_country.apply(
    lambda r: f"{r['country']} ({r['total']:,})", axis=1
)

country_order = (
    top_df.groupby('country').size()
    .sort_values(ascending=False)
    .index.tolist()
)
label_order = [
    sector_by_country.loc[sector_by_country['country'] == c, 'country_label'].iloc[0]
    for c in country_order
]

# Sectors sorted by descending overall share across the full dataset
sector_order = (
    dfc['sector'].value_counts(normalize=True)
    .sort_values(ascending=False)
    .index.tolist()
)

fig_a2 = px.bar(
    sector_by_country,
    x='pct',
    y='country_label',
    color='sector',
    orientation='h',
    custom_data=['pct', 'country'],
    title=f'Sector Distribution for Top {TOP_N_COUNTRIES} Countries by Loan Count',
    labels={
        'pct': '% of Loans',
        'country_label': 'Country',
        'sector': 'Sector'
    },
    category_orders={'country_label': label_order, 'sector': sector_order}
)
fig_a2.update_traces(
    hovertemplate=(
        '<b>%{fullData.name}</b><br>'
        '% of total loans for %{customdata[1]}: %{customdata[0]:.1f}%'
        '<extra></extra>'
    )
)
fig_a2.update_layout(xaxis_ticksuffix='%')
fig_a2.show()

In [ ]:
# Minimum number of loans for a country to appear in the chart -- adjust as needed
MIN_LOANS = 10000

country_bubble = (
    dfc.groupby('country')
    .agg(
        loan_count=('loan_amount', 'count'),
        median_loan_amount=('loan_amount', 'median'),
        total_funded_amount=('funded_amount', 'sum')
    )
    .reset_index()
)

# Dominant sector per country: the sector with the most loans in that country
dominant_sector = (
    dfc.groupby(['country', 'sector'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .drop_duplicates('country')
    .set_index('country')['sector']
)
country_bubble['dominant_sector'] = country_bubble['country'].map(dominant_sector)

country_bubble = (
    country_bubble[country_bubble['loan_count'] >= MIN_LOANS]
    .sort_values('loan_count', ascending=False)
    .reset_index(drop=True)
)

print(f"Countries shown (loan count >= {MIN_LOANS:,}): {len(country_bubble)}")
print(f"Countries excluded (below threshold): "
      f"{dfc['country'].nunique() - len(country_bubble)}")
print("Dominant sectors present:", sorted(country_bubble['dominant_sector'].unique()))

fig_a3 = px.scatter(
    country_bubble,
    x='loan_count',
    y='median_loan_amount',
    size='total_funded_amount',
    color='dominant_sector',
    text='country',
    custom_data=['total_funded_amount', 'dominant_sector'],
    title='Loan Activity vs Typical Loan Size by Country',
    labels={
        'loan_count': 'Number of Loans',
        'median_loan_amount': 'Median Loan Amount (USD)',
        'total_funded_amount': 'Total Funded (USD)',
        'dominant_sector': 'Dominant Sector'
    },
    size_max=60
)
fig_a3.update_traces(
    textposition='middle center',
    textfont_size=10,
    hovertemplate=(
        '<b>%{text}</b><br>'
        'Dominant sector: %{customdata[1]}<br>'
        'Loan count: %{x:,}<br>'
        'Median loan amount: $%{y:,.0f}<br>'
        'Total funded: $%{customdata[0]:,.0f}'
        '<extra></extra>'
    )
)
fig_a3.update_layout(height=750)
fig_a3.show()

---

## Section 4: Dashboard: Sector Lens

A second interactive dashboard is provided for deeper exploration.
For each selected sector, the top X countries are plotted by loan activity
vs median loan size. Bubble area encodes total funded amount; bubble color
encodes the dominant repayment interval in that country for the selected sector.

After this cell is run, **http://localhost:8051** can be opened in the browser.
If Dashboard 1 is still running on port 8050, both can coexist.

#### Port 8051 Check

Any process already listening on port 8051 is terminated before Dashboard 2 is started.

In [ ]:
# kill_port(8051)


In [ ]:
# Dashboard 2 color scheme
BLUE_DEEP2   = '#1d4ed8'
BLUE_MID2    = '#3b82f6'
SIDEBAR_BG2  = '#dbeafe'
LABEL_COLOR2 = '#1e40af'

# Fixed color map for repayment intervals -- consistent across all sector selections
INTERVAL_COLORS = {
    'bullet':        "#f0c714",
    'irregular':     "#f91640",
    'monthly':       "#50e185",
    'weekly':        '#3b82f6',
    'Not specified': '#94a3b8'
}

dash2_df = dfc.copy()
sectors2 = sorted(dash2_df['sector'].unique())
dropdown_options2 = [{'label': 'Overall (all sectors)', 'value': 'Overall'}] + \
                    [{'label': s, 'value': s} for s in sectors2]

app2 = dash.Dash(__name__ + '_2',
    external_stylesheets=[
        'https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap'
    ]
)

app2.layout = html.Div([

    # Sidebar
    html.Div([
        html.H2('Kiva Sector Lens', style={
            'color': BLUE_DEEP2,
            'fontFamily': "'Inter', sans-serif",
            'fontSize': '22px',
            'marginTop': '0',
            'marginBottom': '24px'
        }),
        html.Label('Select Sector:', style={
            'color': LABEL_COLOR2,
            'fontFamily': "'Inter', sans-serif",
            'fontSize': '13px',
            'fontWeight': 'bold',
            'display': 'block',
            'marginBottom': '6px'
        }),
        dcc.Dropdown(
            id='d2-sector',
            options=dropdown_options2,
            value='Overall',
            clearable=False,
            style={'fontFamily': "'Inter', sans-serif", 'fontSize': '14px'}
        ),
        html.Div(style={'height': '16px'}),
        html.Label('Top N countries:', style={
            'color': LABEL_COLOR2,
            'fontFamily': "'Inter', sans-serif",
            'fontSize': '13px',
            'fontWeight': 'bold',
            'display': 'block',
            'marginBottom': '6px'
        }),
        dcc.Input(
            id='d2-top-n',
            type='number',
            value=15,
            min=3,
            max=50,
            step=1,
            style={
                'width': '100%',
                'fontFamily': "'Inter', sans-serif",
                'fontSize': '14px',
                'padding': '6px 8px',
                'border': '1px solid #93c5fd',
                'borderRadius': '4px'
            }
        ),
        html.Hr(style={'borderColor': BLUE_MID2, 'margin': '24px 0'}),
        html.H4('Analysis', style={
            'color': BLUE_DEEP2,
            'fontFamily': "'Inter', sans-serif",
            'fontSize': '16px',
            'marginBottom': '10px'
        }),
        html.P(
            '[Placeholder -- analysis text to be added here]',
            style={
                'color': '#64748b',
                'fontFamily': "'Inter', sans-serif",
                'fontSize': '13px',
                'lineHeight': '1.6',
                'fontStyle': 'italic'
            }
        )
    ], style={
        'flex': '0 0 270px',
        'backgroundColor': SIDEBAR_BG2,
        'padding': '28px 20px',
        'borderRight': f'2px solid {BLUE_MID2}'
    }),

    # Chart
    html.Div(
        dcc.Graph(id='d2-scatter'),
        style={
            'flex': '1',
            'backgroundColor': '#f8fafc',
            'padding': '8px'
        }
    )

], style={
    'display': 'flex',
    'fontFamily': "'Inter', sans-serif",
    'maxWidth': '1400px',
    'margin': '20px auto',
    'boxShadow': '0 4px 14px rgba(0,0,0,0.12)',
    'borderRadius': '6px',
    'overflow': 'hidden',
    'minHeight': '680px'
})


@app2.callback(
    Output('d2-scatter', 'figure'),
    Input('d2-sector', 'value'),
    Input('d2-top-n', 'value')
)
def update_sector_scatter(selected_sector, top_n_raw):
    top_n = max(3, min(50, int(top_n_raw or 15)))

    if selected_sector == 'Overall':
        sector_data = dash2_df
        chart_title = f'Loan Activity vs Median Loan Size -- All Sectors (Top {top_n} Countries)'
    else:
        sector_data = dash2_df[dash2_df['sector'] == selected_sector]
        chart_title = f'Loan Activity vs Median Loan Size -- {selected_sector} (Top {top_n} Countries)'

    country_agg = (
        sector_data.groupby('country')
        .agg(
            loan_count=('loan_amount', 'count'),
            median_loan_amount=('loan_amount', 'median'),
            mean_loan_amount=('loan_amount', 'mean'),
            total_funded_amount=('funded_amount', 'sum')
        )
        .reset_index()
    )

    dominant_interval = (
        sector_data.groupby(['country', 'repayment_interval'])
        .size()
        .reset_index(name='cnt')
        .sort_values('cnt', ascending=False)
        .drop_duplicates('country')
        .set_index('country')['repayment_interval']
    )
    country_agg['dominant_interval'] = country_agg['country'].map(dominant_interval)

    country_agg = (
        country_agg.nlargest(top_n, 'loan_count')
        .reset_index(drop=True)
    )

    fig = px.scatter(
        country_agg,
        x='loan_count',
        y='median_loan_amount',
        size='total_funded_amount',
        color='dominant_interval',
        color_discrete_map=INTERVAL_COLORS,
        text='country',
        custom_data=['total_funded_amount', 'dominant_interval', 'mean_loan_amount'],
        title=chart_title,
        labels={
            'loan_count': 'Number of Loans',
            'median_loan_amount': 'Median Loan Amount (USD)',
            'total_funded_amount': 'Total Funded (USD)',
            'dominant_interval': 'Dominant Repayment Interval'
        },
        size_max=60
    )
    fig.update_traces(
        textposition='middle center',
        textfont_size=10,
        hovertemplate=(
            '<b>%{text}</b><br>'
            'Dominant interval: %{customdata[1]}<br>'
            'Loan count: %{x:,}<br>'
            'Median loan amount: $%{y:,.0f}<br>'
            'Mean loan amount: $%{customdata[2]:,.0f}<br>'
            'Total funded: $%{customdata[0]:,.0f}'
            '<extra></extra>'
        )
    )
    fig.update_layout(
        template='plotly_white',
        plot_bgcolor='#eff6ff',
        title_font_size=16,
        title_font_family='Inter, sans-serif',
        height=620
    )
    return fig


app2.run(debug=False, port=8051, jupyter_mode='external')
